# SL-5 : Integration Neuro-Symbolique

**Navigation** : [Index](./README.md) | [<< SL-4](SL-4-InductiveLogicProgramming.ipynb) | [Suivant >>](SL-6-KnowledgeGraphs-ILP.ipynb)

## Objectifs d'apprentissage

A la fin de ce notebook, vous saurez :
1. Comprendre les motivations de l'integration neuro-symbolique
2. Implementer des operateurs logiques differentiables (t-norms)
3. Construire un reseau de predicats neuronaux simplifie
4. Appliquer le raisonnement guide par la logique sur un reseau de neurones
5. Implementer un mini-systeme DeepProbLog

### Prerequis
- SL-1 a SL-4 completes
- Notions de base en numpy (operations matricielles)
- Python 3.10+ (numpy uniquement)

### Duree estimee : 50-60 minutes

---

L'integration neuro-symbolique combine les **reseaux de neurones** (apprentissage a partir de donnees) avec le **raisonnement symbolique** (logique, regles). L'objectif est d'obtenir le meilleur des deux mondes.

## 1. Pourquoi combiner neural et symbolique ?

| Force | Neuronal | Symbolique |
|-------|----------|------------|
| Apprentissage a partir de donnees | Excellent | Impossible |
| Raisonnement formel | Aucune garantie | Preuves mathematiques |
| Tolerance au bruit | Elevee | Faible |
| Interpretation | Boite noire | Transparent |
| Generalisation | Interpolative | Deductive |

In [1]:
import numpy as np
from typing import List, Tuple, Callable

np.random.seed(42)  # reproductibilite des initialisations aleatoires

print('=== SL-5 : Integration Neuro-Symbolique ===')
print(f'NumPy version : {np.__version__}')

=== SL-5 : Integration Neuro-Symbolique ===
NumPy version : 1.26.0


### Interpretation : Environnement

Ce notebook utilise uniquement numpy pour les calculs. Aucune bibliotheque d'apprentissage profond n'est requise : nous implementons les concepts fondamentaux a la main pour bien les comprendre.

## 2. Operateurs logiques differentiables

En logique classique, les connecteurs sont binaires (vrai/faux). En logique **floue** ou **differentiable**, on remplace ces valeurs binaires par des nombres dans [0, 1]. Les t-norms et t-conorms sont les generalisations differentiables de AND et OR.

| Connecteur | Logique classique | Version differentiable (utilisee ici) |
|------------|-------------------|---------------------------------------|
| AND(a, b) | min(a, b) | a * b (**t-norm produit**) |
| OR(a, b) | max(a, b) | a + b - a*b (t-conorm probabiliste) |
| NOT(a) | 1 - a | 1 - a |
| IMPLIES(a, b) | NOT(a) OR b | min(1, b/a) (**implication de Goguen**, residuelle du produit) |

> Il existe plusieurs familles de t-norms : **produit** (a*b), **Godel** (min(a,b),
> differentiable presque partout) et **Lukasiewicz** (max(0, a+b-1), cf. exercice 4).
> Chaque t-norm induit son implication residuelle ; celle du produit est Goguen.

In [2]:
def fuzzy_and(a: float, b: float) -> float:
    return a * b

def fuzzy_or(a: float, b: float) -> float:
    return a + b - a * b

def fuzzy_not(a: float) -> float:
    return 1.0 - a

def fuzzy_implies(a: float, b: float) -> float:
    return min(1.0, b / max(a, 1e-10))

# Demonstration
print('=== Tables de verite floues ===')
print('a\tb\tAND\tOR\tNOT(a)\tIMPLIES')
print('-' * 50)
for a in [0.0, 0.3, 0.5, 0.7, 1.0]:
    for b in [0.0, 0.5, 1.0]:
        print(f'{a:.1f}\t{b:.1f}\t{fuzzy_and(a,b):.2f}\t{fuzzy_or(a,b):.2f}\t{fuzzy_not(a):.2f}\t{fuzzy_implies(a,b):.2f}')
    print()

=== Tables de verite floues ===
a	b	AND	OR	NOT(a)	IMPLIES
--------------------------------------------------
0.0	0.0	0.00	0.00	1.00	0.00
0.0	0.5	0.00	0.50	1.00	1.00
0.0	1.0	0.00	1.00	1.00	1.00

0.3	0.0	0.00	0.30	0.70	0.00
0.3	0.5	0.15	0.65	0.70	1.00
0.3	1.0	0.30	1.00	0.70	1.00

0.5	0.0	0.00	0.50	0.50	0.00
0.5	0.5	0.25	0.75	0.50	1.00
0.5	1.0	0.50	1.00	0.50	1.00

0.7	0.0	0.00	0.70	0.30	0.00
0.7	0.5	0.35	0.85	0.30	0.71
0.7	1.0	0.70	1.00	0.30	1.00

1.0	0.0	0.00	1.00	0.00	0.00
1.0	0.5	0.50	1.00	0.00	0.50
1.0	1.0	1.00	1.00	0.00	1.00



### Interpretation : Operateurs differentiables

Les operateurs flous transforment les connecteurs logiques binaires en fonctions continues derivables :
- **AND(a,b) = a*b** : la verite de la conjonction est le produit des verites
- **OR(a,b) = a+b-a*b** : la disjonction probabiliste
- **IMPLIES(a,b)** : vaut 1 si b >= a (l'implication est satisfaite)

Ces fonctions sont derivables, ce qui permet de les utiliser dans des reseaux de neurones.

## 3. Predicats neuronaux

Un **predicat neuronal** est une fonction P(x1, ..., xn) -> [0, 1] implementee par un petit reseau de neurones. Il associe une valeur de verite continue a un tuple d'entrees.

In [3]:
class NeuralPredicate:
    """Un predicat neuronal : P(x) -> [0, 1]."""
    
    def __init__(self, n_inputs: int, name: str = 'P'):
        self.name = name
        self.weights = np.random.randn(n_inputs) * 0.1
        self.bias = np.random.randn() * 0.1
    
    def __call__(self, x: np.ndarray) -> float:
        z = np.dot(self.weights, x) + self.bias
        return 1.0 / (1.0 + np.exp(-z))
    
    def gradient(self, x: np.ndarray) -> Tuple[np.ndarray, float]:
        val = self(x)
        d_sigmoid = val * (1 - val)
        return d_sigmoid * x, d_sigmoid
    
    def update(self, grad_w: np.ndarray, grad_b: float, lr: float = 0.1):
        self.weights += lr * grad_w
        self.bias += lr * grad_b

# Test
parent_pred = NeuralPredicate(n_inputs=2, name='parent')
x_test = np.array([1.0, 0.5])
print(f'parent({x_test[0]:.1f}, {x_test[1]:.1f}) = {parent_pred(x_test):.4f}')

parent(1.0, 0.5) = 0.5269


### Interpretation : Predicats neuronaux

Chaque predicat neuronal est un petit classificateur logistique. Initialement, les poids sont aleatoires, donc les predictions sont proches de 0.5. L'entrainement va ajuster ces poids.

## 4. Logique Tensorielle (LTN) simplifiee

Les **Logical Tensor Networks** (LTN) combinent des predicats neuronaux avec des connecteurs logiques differentiables. L'idee est de maximiser la verite globale d'un ensemble de formules logiques.

In [4]:
# Encodage des individus
entities = {'Marie': 0, 'Pierre': 1, 'Paul': 2, 'Sophie': 3, 'Luc': 4}
entity_embeddings = np.eye(len(entities), dtype=float)

def get_embedding(name: str) -> np.ndarray:
    return entity_embeddings[entities[name]]

def pair_embedding(s: str, o: str) -> np.ndarray:
    """Concatenation des one-hot du sujet et de l'objet (2 x 5 = 10 entrees)."""
    return np.concatenate([get_embedding(s), get_embedding(o)])

# Entrainement LTN
parent_pred = NeuralPredicate(n_inputs=2 * len(entities), name='parent')

positive = [('Marie', 'Pierre'), ('Pierre', 'Paul'), ('Sophie', 'Luc')]
# Negatifs echantillonnes en monde clos (comme pour les embeddings de KG, cf. SL-6)
negative = [('Marie', 'Paul'), ('Pierre', 'Sophie'), ('Sophie', 'Pierre'),
            ('Paul', 'Marie'), ('Luc', 'Luc'), ('Luc', 'Sophie')]

print('=== Entrainement du predicat neuronal "parent" ===')
# Gradient de l'entropie croisee : pour une sortie sigmoide, la derivee de la
# perte par rapport au score z se simplifie en (val - cible) --- la derivee de
# la sigmoide se compense. D'ou la mise a jour w += lr * (cible - val) * x.
lr = 0.5
for epoch in range(150):
    total_loss = 0.0
    for s, o in positive:
        x = pair_embedding(s, o)
        val = parent_pred(x)
        total_loss += -np.log(max(val, 1e-10))
        parent_pred.update(x * (1 - val), (1 - val), lr)
    for s, o in negative:
        x = pair_embedding(s, o)
        val = parent_pred(x)
        total_loss += -np.log(max(1 - val, 1e-10))
        parent_pred.update(-x * val, -val, lr)
    if (epoch + 1) % 30 == 0:
        print(f'  Epoch {epoch+1:2d} : perte = {total_loss:.4f}')

print('\n=== Resultats (cible : positifs proches de 1, negatifs proches de 0) ===')
for s, o in positive:
    print(f'  parent({s}, {o}) = {parent_pred(pair_embedding(s, o)):.4f}   [positif]')
for s, o in negative:
    print(f'  parent({s}, {o}) = {parent_pred(pair_embedding(s, o)):.4f}   [negatif]')


=== Entrainement du predicat neuronal "parent" ===
  Epoch 30 : perte = 2.3809
  Epoch 60 : perte = 1.3391
  Epoch 90 : perte = 0.9147
  Epoch 120 : perte = 0.6909
  Epoch 150 : perte = 0.5539

=== Resultats (cible : positifs proches de 1, negatifs proches de 0) ===
  parent(Marie, Pierre) = 0.8966   [positif]
  parent(Pierre, Paul) = 0.9193   [positif]
  parent(Sophie, Luc) = 0.9143   [positif]
  parent(Marie, Paul) = 0.0746   [negatif]
  parent(Pierre, Sophie) = 0.0353   [negatif]
  parent(Sophie, Pierre) = 0.0804   [negatif]
  parent(Paul, Marie) = 0.0032   [negatif]
  parent(Luc, Luc) = 0.0391   [negatif]
  parent(Luc, Sophie) = 0.0000   [negatif]


### Interpretation : Entrainement LTN

Le predicat neuronal `parent` apprend a distinguer les vraies relations parentales
(valeur proche de 1) des fausses (valeur proche de 0). L'entrainement utilise la
descente de gradient classique sur une perte de type entropie croisee.

Deux choix de representation sont importants :

- **Encodage des paires** : on concatene les one-hot du sujet et de l'objet
  (10 entrees). Le classificateur logistique apprend alors un score additif
  *w(sujet) + w(objet)* --- suffisant ici car les exemples sont lineairement
  separables dans cette representation.
- **Negatifs en monde clos** : comme aucune base ne fournit de "non-parents"
  explicites, on echantillonne des paires absentes des faits positifs. C'est
  exactement la strategie d'entrainement des embeddings de Knowledge Graphs (SL-6).

| Avant entrainement | Apres entrainement (60 epoques) |
|-------------------|--------------------------------|
| Valeurs ~0.5 (aleatoire) | Positifs ~0.9, negatifs < 0.1 |


## 5. Raisonnement guide par la logique

En Neuro-Symbolique, on peut utiliser des **regles logiques pour guider l'entrainement** d'un reseau de neurones. Par exemple : si `parent(X,Y) AND parent(Y,Z)`, alors `grandparent(X,Z)` devrait etre vrai.

In [5]:
grandparent_pred = NeuralPredicate(n_inputs=2 * len(entities), name='grandparent')
chains = [('Marie', 'Pierre', 'Paul')]

print('=== Entrainement guide par regle ===')
print('Regle : parent(X,Y) AND parent(Y,Z) => grandparent(X,Z)')

lr = 0.5
for epoch in range(30):
    total_loss = 0.0
    for g, p, c in chains:
        x_gc = pair_embedding(g, c)
        gp_val = parent_pred(pair_embedding(g, p))
        pc_val = parent_pred(pair_embedding(p, c))
        gc_val = grandparent_pred(x_gc)
        body = fuzzy_and(gp_val, pc_val)
        rule_truth = fuzzy_implies(body, gc_val)
        loss = -np.log(max(rule_truth, 1e-10))
        total_loss += loss
        gw, gb = grandparent_pred.gradient(x_gc)
        error = body - gc_val
        grandparent_pred.update(gw * error, gb * error, lr)
    if (epoch + 1) % 10 == 0:
        print(f'  Epoch {epoch+1:2d} : perte = {total_loss:.4f}')

print('\n=== Verifications ===')
x_gc = pair_embedding('Marie', 'Paul')
print(f'grandparent(Marie, Paul) = {grandparent_pred(x_gc):.4f}')
print(f'  (Attendu : eleve, proche de la verite du corps de la regle ~0.8)')
x_sl = pair_embedding('Sophie', 'Luc')
print(f'grandparent(Sophie, Luc) = {grandparent_pred(x_sl):.4f}')
print(f'  (Attendu : ~0.5 --- paire non couverte par la regle, donc non contrainte)')

=== Entrainement guide par regle ===
Regle : parent(X,Y) AND parent(Y,Z) => grandparent(X,Z)
  Epoch 10 : perte = 0.2284
  Epoch 20 : perte = 0.1096
  Epoch 30 : perte = 0.0637

=== Verifications ===
grandparent(Marie, Paul) = 0.7757
  (Attendu : eleve, proche de la verite du corps de la regle ~0.8)
grandparent(Sophie, Luc) = 0.5821
  (Attendu : ~0.5 --- paire non couverte par la regle, donc non contrainte)


### Interpretation : Raisonnement guide

Le predicat `grandparent` a ete entraine **sans exemples directs** de grand-parent. Il a appris uniquement a partir de la regle logique `parent(X,Y) AND parent(Y,Z) => grandparent(X,Z)`.

C'est la puissance du neuro-symbolique : la **connaissance logique** guide l'apprentissage neuronal, meme sans donnees etiquetees pour le predicat cible.
> **Limite a noter** : la regle est un *modus ponens* flou --- elle ne fournit
> qu'une contrainte **positive** (si le corps est vrai, la tete doit l'etre).
> Pour la paire (Sophie, Luc), aucune chaine parent-parent n'existe : le predicat
> reste a son initialisation (~0.5), ni vrai ni faux. Pousser ces paires vers 0
> demanderait des exemples negatifs ou une hypothese de monde clos.


## 6. DeepProbLog simplifie

**DeepProbLog** combine la programmation logique probabiliste avec des predicats neuronaux.

In [6]:
from itertools import product as iter_product

class SimpleDeepProbLog:
    def __init__(self):
        self.neural_preds = {}
        self.rules = []
        self.facts = {}

    def add_neural_predicate(self, name, pred):
        self.neural_preds[name] = pred

    def add_rule(self, body, head):
        self.rules.append((body, head))

    def add_fact(self, predicate, args, prob):
        self.facts[(predicate, args)] = prob

    def _is_variable(self, arg):
        return len(arg) == 1 and arg.isupper()

    def query(self, predicate, args, depth=0, max_depth=5):
        """Chainage arriere : probabilite que predicate(args) soit vrai.

        Les variables du corps non liees par la tete sont EXISTENTIELLES :
        on enumere les individus et on garde le meilleur binding (max).
        """
        if depth > max_depth:
            return 0.0
        key = (predicate, args)
        if key in self.facts:
            return self.facts[key]
        if predicate in self.neural_preds:
            x = np.concatenate([get_embedding(a) for a in args])
            return float(self.neural_preds[predicate](x))
        max_prob = 0.0
        for body, head in self.rules:
            if head[0] != predicate or len(head[1]) != len(args):
                continue
            subst = {}
            match = True
            for h_a, q_a in zip(head[1], args):
                if self._is_variable(h_a):
                    subst[h_a] = q_a
                elif h_a != q_a:
                    match = False
                    break
            if not match:
                continue
            free = sorted({a for _, b_args in body for a in b_args
                           if self._is_variable(a) and a not in subst})
            for binding in iter_product(entities, repeat=len(free)):
                s = dict(subst)
                s.update(zip(free, binding))
                body_truth = 1.0
                for b_pred, b_args in body:
                    resolved = tuple(s.get(a, a) for a in b_args)
                    body_truth = fuzzy_and(
                        body_truth, self.query(b_pred, resolved, depth + 1, max_depth))
                    if body_truth == 0.0:
                        break
                max_prob = max(max_prob, body_truth)
        return max_prob

dpl = SimpleDeepProbLog()
dpl.add_neural_predicate('parent', parent_pred)
dpl.add_rule([('parent', ('X', 'Y')), ('parent', ('Y', 'Z'))], ('grandparent', ('X', 'Z')))

print('=== Requetes DeepProbLog ===')
for pred, args in [('parent', ('Marie', 'Pierre')), ('parent', ('Marie', 'Paul')),
                   ('grandparent', ('Marie', 'Paul')), ('grandparent', ('Sophie', 'Luc'))]:
    prob = dpl.query(pred, args)
    print(f'  {pred}{args} = {prob:.4f}')


=== Requetes DeepProbLog ===
  parent('Marie', 'Pierre') = 0.8966
  parent('Marie', 'Paul') = 0.0746
  grandparent('Marie', 'Paul') = 0.8242
  grandparent('Sophie', 'Luc') = 0.0804


### Interpretation : DeepProbLog

Le systeme DeepProbLog combine :
1. **Predicats neuronaux** : `parent` est le predicat appris en section 4
2. **Regles logiques** : `grandparent(X,Z) :- parent(X,Y), parent(Y,Z)`
3. **Inference** : la requete `grandparent(Marie, Paul)` est resolue par
   **chainage arriere**. La variable `Y` du corps n'est pas liee par la tete :
   elle est **existentielle**. Le moteur enumere donc les 5 individus et garde
   le meilleur binding --- ici `Y = Pierre`, qui donne
   `parent(Marie, Pierre) * parent(Pierre, Paul)`, un produit eleve.

La probabilite du corps est le **produit** des probabilites de ses atomes
(t-norm produit), et la quantification existentielle est approximee par le
**max** sur les bindings. `grandparent(Sophie, Luc)` reste faible : aucune
chaine `parent-parent` plausible ne relie Sophie a Luc en deux pas.


## 7. Comparaison des approches neuro-symboliques

| Approche | Principe | Forces | Limites |
|----------|----------|--------|----------|
| **LTN** | Predicats neuronaux + semantique logique | Theoriquement fonde | Complexe a entrainer |
| **Neural Theorem Prover** | Unification differentiable | Interpretable | Passage a l'echelle |
| **DeepProbLog** | Problog + predicats neuronaux | Flexible | Complexite combinatoire |
| **EBL Neural** | Explications differentiables | Efficace | Domaine-specifique |

## Exercices et exemple guidé

Cette section illustre les concepts par la pratique. Elle contient :

- **Exemple guidé 1** : opérateur OR différentiable paramétrique (solution démontrée — bascule #1455).
- **Exercice 2** : entraînement LTN d'un prédicat `frere` puis dérivation d'`oncle`.
- **Exercice 3** : règle transitive `ancestor` dans DeepProbLog.
- **Exercice 4** *(nouveau, ajouté avec la bascule #1455)* : t-norm de Łukasiewicz et famille paramétrique à 3 t-conorms.

### Exemple guidé 1 : Opérateur OR différentiable paramétrique

> **Bascule TP étudiant → Exemple guidé** (#1455). Solution démontrée ci-dessous.

#### Objectif

Construire une fonction `parametric_or(a, b, alpha)` qui **interpole** entre deux opérateurs OR flous classiques :

| `alpha` | Forme | Interprétation |
|---------|-------|---------------|
| `0.0`   | `max(a, b)` (Zadeh / Gödel t-conorm) | OR optimiste : "au moins l'un" |
| `1.0`   | `a + b - a * b` (Goguen / probabiliste) | OR indépendant : événements probabilistes |
| `0 < alpha < 1` | combinaison convexe des deux | t-conorm paramétrique |

L'intérêt pédagogique : le paramètre `alpha` devient **différentiable**, donc *apprenable* dans un réseau neuro-symbolique. On peut laisser le réseau choisir entre les deux sémantiques selon la tâche.

#### Solution démontrée

La formule est une combinaison convexe :

$$\text{parametric\_or}(a, b, \alpha) = \alpha \cdot (a + b - a \cdot b) + (1 - \alpha) \cdot \max(a, b)$$

Propriétés vérifiées dans la cellule code suivante :
- **Bornes** : `parametric_or(0, 0, alpha) = 0` et `parametric_or(1, 1, alpha) = 1`
- **Continuité en alpha** : le passage de `max` à `a+b-ab` est lisse
- **Monotonie** : `parametric_or` est croissante en `a` (et en `b`) pour tout `alpha ∈ [0, 1]`

In [7]:
# Exemple guide 1 : Operateur OR parametrique
# Bascule TP -> Exemple guide (#1455) : solution demontree ci-dessous.

# Indice initial conserve a titre pedagogique :
#   parametric_or(a, b, alpha) = alpha * (a + b - a*b) + (1 - alpha) * max(a, b)

def parametric_or(a: float, b: float, alpha: float) -> float:
    """OR flou parametrique interpolant Godel (max) et probabiliste (a+b-a*b).

    Parameters
    ----------
    a, b : float in [0, 1]
        Verites floues des deux propositions.
    alpha : float in [0, 1]
        Poids d'interpolation. alpha=0 -> max(a, b) (Godel),
        alpha=1 -> a + b - a*b (probabiliste).

    Returns
    -------
    float in [0, 1]
        Combinaison convexe des deux t-conorms.
    """
    return alpha * (a + b - a * b) + (1.0 - alpha) * max(a, b)


# Etape 1 : verification des bornes
print('=== Verification des bornes ===')
for alpha in [0.0, 0.5, 1.0]:
    v00 = parametric_or(0.0, 0.0, alpha)
    v11 = parametric_or(1.0, 1.0, alpha)
    print(f'  alpha={alpha:.2f}  parametric_or(0,0)={v00:.4f}   parametric_or(1,1)={v11:.4f}')

# Etape 2 : interpolation lisse entre Godel (alpha=0) et probabiliste (alpha=1)
print('\n=== Sweep alpha pour (a, b) = (0.3, 0.6) ===')
print('alpha\tparametric_or\tcomparaison')
print('-' * 60)
ref_godel = max(0.3, 0.6)
ref_prob = 0.3 + 0.6 - 0.3 * 0.6
for alpha in [0.0, 0.25, 0.5, 0.75, 1.0]:
    val = parametric_or(0.3, 0.6, alpha)
    if alpha == 0.0:
        tag = f'= max(0.3, 0.6) = {ref_godel:.4f}'
    elif alpha == 1.0:
        tag = f'= 0.3+0.6-0.3*0.6 = {ref_prob:.4f}'
    else:
        tag = f'(entre {ref_godel:.4f} et {ref_prob:.4f})'
    print(f'{alpha:.2f}\t{val:.4f}\t\t{tag}')

# Etape 3 : monotonie en a (verification simple)
print('\n=== Monotonie en a (alpha=0.5, b=0.4) ===')
for a in [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]:
    print(f'  parametric_or({a:.1f}, 0.4, 0.5) = {parametric_or(a, 0.4, 0.5):.4f}')

print('\nExemple guide 1 OK : parametric_or implemente, bornes verifiees, sweep alpha affiche.')

=== Verification des bornes ===
  alpha=0.00  parametric_or(0,0)=0.0000   parametric_or(1,1)=1.0000
  alpha=0.50  parametric_or(0,0)=0.0000   parametric_or(1,1)=1.0000
  alpha=1.00  parametric_or(0,0)=0.0000   parametric_or(1,1)=1.0000

=== Sweep alpha pour (a, b) = (0.3, 0.6) ===
alpha	parametric_or	comparaison
------------------------------------------------------------
0.00	0.6000		= max(0.3, 0.6) = 0.6000
0.25	0.6300		(entre 0.6000 et 0.7200)
0.50	0.6600		(entre 0.6000 et 0.7200)
0.75	0.6900		(entre 0.6000 et 0.7200)
1.00	0.7200		= 0.3+0.6-0.3*0.6 = 0.7200

=== Monotonie en a (alpha=0.5, b=0.4) ===
  parametric_or(0.0, 0.4, 0.5) = 0.4000
  parametric_or(0.2, 0.4, 0.5) = 0.4600
  parametric_or(0.4, 0.4, 0.5) = 0.5200
  parametric_or(0.6, 0.4, 0.5) = 0.6800
  parametric_or(0.8, 0.4, 0.5) = 0.8400
  parametric_or(1.0, 0.4, 0.5) = 1.0000

Exemple guide 1 OK : parametric_or implemente, bornes verifiees, sweep alpha affiche.


### Exercice 2 : Entrainement LTN sur un nouveau domaine

Entrainez un predicat neuronal `frere(X,Y)` sur des faits positifs et negatifs, puis utilisez la regle `frere(X,Y) AND parent(Y,Z) => oncle(X,Z)` pour apprendre `oncle`.

In [8]:
# Exercice 2 : Entrainement LTN pour frere et oncle
# TODO etudiant : creez et entrainez les predicats frere et oncle

# Indice : suivez le meme pattern que l'entrainement parent/grandparent

print('Exercice a completer : entrainez frere et oncle')

Exercice a completer : entrainez frere et oncle


### Exercice 3 : Regle transitive ancestor

Ajoutez une regle transitive `ancestor(X,Z) :- parent(X,Y), ancestor(Y,Z)` au systeme DeepProbLog, avec le fait de base `ancestor(X,Y) :- parent(X,Y)`.

In [9]:
# Exercice 3 : Regle transitive ancestor
# TODO etudiant : ajoutez les regles ancestor au systeme dpl

# Indice : ajoutez 2 regles :
#   1. parent(X,Y) => ancestor(X,Y)
#   2. parent(X,Y) AND ancestor(Y,Z) => ancestor(X,Z)
# Attention a la limite de profondeur pour eviter les boucles infinies

print('Exercice a completer : ajoutez la regle ancestor transitive')

Exercice a completer : ajoutez la regle ancestor transitive


### Exercice 4 : T-norm de Łukasiewicz et famille paramétrique à 3 t-conorms

> **Nouvel exercice** (#1455). Ajouté à la suite de l'Exemple guidé 1, comme prolongement de l'opérateur OR paramétrique.

#### Contexte

La t-norm probabiliste (`a * b`) et la t-norm de Gödel / min-max ne sont pas les seules. La **t-norm de Łukasiewicz** est une troisième famille classique, particulièrement utilisée en logique floue car elle satisfait l'**involution de la négation** (`NOT NOT a = a`) tout en restant différentiable presque partout.

| Opérateur | Définition |
|-----------|------------|
| `AND_L(a, b)` | `max(0, a + b − 1)` |
| `OR_L(a, b)`  | `min(1, a + b)` |

#### Tâche

1. **Étape 1** : implémenter `lukasiewicz_and` et `lukasiewicz_or` dans la cellule code suivante (1 ligne chacune).
2. **Étape 2** : vérifier que les bornes `(0,0)` et `(1,1)` donnent bien `0` et `1` respectivement.
3. **Étape 3** : généraliser l'**Exemple guidé 1** en une famille paramétrique à **deux paramètres** `alpha`, `beta` qui combine **3 t-conorms** : probabiliste, max (Gödel) et Łukasiewicz. La combinaison doit rester une combinaison convexe (les poids somment à 1).
4. **Étape 4** : produire un **sweep** sur `alpha ∈ {0, 0.25, 0.5, 0.75, 1.0}` pour `(a=0.3, b=0.6)` en réutilisant `parametric_or` de l'Exemple guidé 1 (mono-paramètre).
5. **Étape 5 (bonus)** : explorer la surface `parametric_or_family(0.3, 0.6, alpha, beta)` sur une grille `numpy` et identifier les `(alpha, beta)` qui maximisent la valeur OR.

#### Pistes pédagogiques

- L'**Étape 1** est très courte. Elle force à manipuler les bornes `[0, 1]` correctement avec `max` et `min`.
- L'**Étape 3** est conceptuelle : si l'on a 3 opérateurs, une combinaison convexe à 2 paramètres `(alpha, beta)` avec `alpha + beta ≤ 1` couvre tout le simplexe.
- L'**Étape 5** prépare au concept de **t-conorm apprenable** : dans un réseau neuro-symbolique réel, `(alpha, beta)` seraient des paramètres entraînés par descente de gradient.

In [10]:
# Exercice 4 : T-norm de Lukasiewicz + sweep alpha
# TODO etudiant : implementer les fonctions ci-dessous et produire le sweep

# Etape 1 : implementer la t-norm de Lukasiewicz et sa t-conorm associee
#   AND_L(a, b) = max(0, a + b - 1)
#   OR_L(a, b)  = min(1, a + b)
def lukasiewicz_and(a: float, b: float) -> float:
    return None  # TODO etudiant : remplacer par max(0.0, a + b - 1.0)

def lukasiewicz_or(a: float, b: float) -> float:
    return None  # TODO etudiant : remplacer par min(1.0, a + b)

# Etape 2 : verifier les bornes (a=0,b=0) et (a=1,b=1) sur AND_L et OR_L
# Indice : utiliser un assert pour valider lukasiewicz_and(0.0, 0.0) == 0.0 etc.

# Etape 3 : ecrire une famille parametrique 3 t-conorms reposant sur l'Exemple guide 1
#   parametric_or_family(a, b, alpha, beta) qui combine
#     - probabiliste (a+b-a*b),
#     - max (Godel),
#     - Lukasiewicz (min(1, a+b)).
#   Indice : utiliser deux poids alpha, beta dans [0, 1] tels que alpha + beta <= 1.
def parametric_or_family(a: float, b: float, alpha: float, beta: float) -> float:
    return None  # TODO etudiant : combinaison convexe des 3 OR

# Etape 4 : produire un sweep sur alpha in {0.0, 0.25, 0.5, 0.75, 1.0} pour (a=0.3, b=0.6)
# Indice : afficher un petit tableau alpha -> parametric_or(0.3, 0.6, alpha) en utilisant
#          la fonction parametric_or de l'Exemple guide 1 ci-dessus.

# Etape 5 (bonus) : tracer numpy uniquement (np.linspace + print formate)
# de la surface alpha x beta -> parametric_or_family(0.3, 0.6, alpha, beta).
# Quels (alpha, beta) maximisent la verite de OR pour ces valeurs ?

print('Exercice 4 a completer : implementez lukasiewicz_and / lukasiewicz_or / parametric_or_family + sweep')

Exercice 4 a completer : implementez lukasiewicz_and / lukasiewicz_or / parametric_or_family + sweep


---

## 8. Resume

### Points cles

| Concept | Description |
|---------|-------------|
| T-norm / T-conorm | Generalisation differentiable de AND / OR |
| Predicat neuronal | Fonction P(x) -> [0,1] apprise par reseau de neurones |
| LTN | Logique Tensorielle : predicats neuronaux + semantique logique |
| Raisonnement guide | Regles logiques guident l'entrainement neuronal |
| DeepProbLog | Programmation logique probabiliste + predicats neuronaux |

**Ce qu'il faut retenir** :
- La logique differentiable transforme les connecteurs binaires en fonctions continues derivables
- Les predicats neuronaux peuvent etre combines avec des regles logiques
- L'entrainement peut etre guide par la connaissance logique, meme sans donnees etiquetees
- DeepProbLog unifie programmation logique et reseaux de neurones


### Perspectives

Ce notebook a explore l'integration neuro-symbolique, qui vise a combiner les forces complementaires des reseaux de neurones (apprentissage a partir de donnees, tolerance au bruit) et du raisonnement symbolique (garanties formelles, interpretabilite). Les operateurs logiques differentiables (t-norms) transforment les connecteurs binaires AND, OR, NOT en fonctions continues derivables sur [0, 1], permettant leur integration dans des graphes de calcul optimisables par descente de gradient. Les predicats neuronaux implementent des fonctions P(x) -> [0, 1] via des reseaux de neurones, et la Logique Tensorielle (LTN) combine ces predicats avec une semantique logique pour entrainer des modeles guides par des regles.

L'implementation du raisonnement guide par regle a illustre un resultat remarquable : le predicat `grandparent` a ete appris sans aucun exemple direct de grand-parent, uniquement a partir de la regle logique `parent(X,Y) AND parent(Y,Z) => grandparent(X,Z)`. Le systeme DeepProbLog simplifie a ensuite unifie la programmation logique probabiliste et les predicats neuronaux, permettant de resoudre des requetes par chainage arriere. L'exemple guide de l'operateur OR parametrique a demontre comment un parametre de combinaison convexe (alpha) peut etre appris pour choisir automatiquement entre la semantique de Godel (max) et la semantique probabiliste (a+b-a*b).

L'integration neuro-symbolique reste un domaine de recherche actif. Les approches LTN, Neural Theorem Prover et DeepProbLog chacune proposent des compromis differents entre expressivite logique, passage a l'echelle et facilite d'entrainement. Le notebook suivant, [SL-6 - Knowledge Graphs et ILP](SL-6-KnowledgeGraphs-ILP.ipynb), aborde une autre dimension de l'apprentissage symbolique moderne : le minage de regles dans les Knowledge Graphs, ou les triples RDF remplacent les faits logiques traditionnels et l'echelle atteint des milliards de donnees.

---

## Ressources

- Serafini & Garcez, "Logic Tensor Networks" (2016)
- Manhaeve et al., "DeepProbLog: Neural Probabilistic Logic Programming" (2018)
- Rocktaschel & Riedel, "End-to-End Differentiable Proving" (2017)
- Russell & Norvig, *AIMA*, 4e ed., Chapitre 19

**Navigation** : [Index](./README.md) | [<< SL-4 ILP](SL-4-InductiveLogicProgramming.ipynb) | [Suivant >> SL-6 KG-ILP](SL-6-KnowledgeGraphs-ILP.ipynb)